<a href="https://colab.research.google.com/github/jugernaut/MACTI-manejodatos/blob/principal/07_RedesNeuronales/MetodoNewton_interactive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def f(x):
    """La función para la cual encontrar la raíz."""
    return x**2

def df(x):
    return 2*x

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches # Importar 'patches' para dibujar rectángulos
from ipywidgets import interact, FloatSlider, IntSlider
from IPython.display import display

def newton_en_pasos(f, df, Tol, N, x0):
    """
    Método de Newton que devuelve todas las aproximaciones intermedias e información de la línea tangente.
    f: La función.
    df: La derivada de la función.
    Tol: Tolerancia para la convergencia.
    N: Número máximo de iteraciones.
    x0: Aproximación inicial.
    Devuelve: (lista de aproximaciones de x, lista de información de la línea tangente)
    """
    n = 1
    x_approximations = [x0] # Almacenar la aproximación inicial
    tangent_lines_info = [] # Almacenar información para las líneas tangentes: (punto_x, punto_y, pendiente)

    current_x = x0

    while n <= N:
        fx = f(current_x)
        dfx = df(current_x)

        if dfx == 0:
            print(f"Advertencia: La derivada es cero en x = {current_x}. El método de Newton no puede continuar.")
            break

        # Calcular la siguiente aproximación
        next_x = current_x - (fx / float(dfx))

        # Almacenar información de la línea tangente para el punto actual
        tangent_lines_info.append({
            'x_point': current_x,
            'y_point': fx,
            'slope': dfx
        })

        x_approximations.append(next_x)

        # Verificar los criterios de convergencia
        if abs(f(next_x)) <= Tol and abs(next_x - current_x) <= Tol:
            break

        current_x = next_x
        n += 1

    return x_approximations, tangent_lines_info

def plot_newton(iterations, initial_guess):
    """
    Gráfica del método de Newton interactivo.
    iterations: Número de iteraciones.
    initial_guess: Aproximación inicial
    """
    tolerance = 1e-6

    # Obtener aproximaciones e información de las tangentes
    x_approxs, tangents = newton_en_pasos(f, df, tolerance, iterations, initial_guess)

    plt.figure(figsize=(10, 7))

    # Determinar el rango del eje x para el trazado
    # Usar un rango más amplio si las aproximaciones son pocas o muy separadas
    x_min_plot = min(min(x_approxs) if x_approxs else initial_guess, -5.0) - 1
    x_max_plot = max(max(x_approxs) if x_approxs else initial_guess, 5.0) + 1

    # Asegurarse de que la raíz esté dentro del rango visible si se encuentra (para f(x) = x^2 - 2, las raíces son +/- sqrt(2))
    if len(x_approxs) > 1:
        root_approx = x_approxs[-1]
        x_min_plot = min(x_min_plot, root_approx - 2)
        x_max_plot = max(x_max_plot, root_approx + 2)
    else:
        x_min_plot = min(x_min_plot, initial_guess - 2)
        x_max_plot = max(x_max_plot, initial_guess + 2)

    x_vals = np.linspace(x_min_plot, x_max_plot, 400)
    y_vals = f(x_vals)

    # Establecer límites y para ajustar la función y las aproximaciones correctamente
    y_plot_min = y_vals.min()
    y_plot_max = y_vals.max()

    if x_approxs:
        y_approx_vals = [f(x) for x in x_approxs]
        y_plot_min = min(y_plot_min, min(y_approx_vals))
        y_plot_max = max(y_plot_max, max(y_approx_vals))

    # Añadir relleno a los límites y
    y_plot_min -= 1
    y_plot_max += 1

    plt.plot(x_vals, y_vals, label='f(x) = $x^2$', color='blue', linewidth=2)
    plt.axhline(0, color='black', linewidth=1.2, linestyle='-') # eje x
    plt.axvline(0, color='black', linewidth=1.2, linestyle='-') # eje y

    # Trazar aproximaciones y líneas tangentes
    for i in range(len(x_approxs)):
        x_val = x_approxs[i]
        y_val = f(x_val)

        # Trazar punto de aproximación en f(x)
        plt.scatter(x_val, y_val, color='red', s=70, zorder=5,
                    label=f'$x_{i}$ (Aprox. en f(x))' if i == 0 else "", marker='o', edgecolors='black')
        plt.text(x_val, y_val + 0.2, f'$x_{i}$', fontsize=10, ha='center', va='bottom')

        # Trazar línea vertical desde la aproximación hasta el eje x
        plt.plot([x_val, x_val], [0, y_val], color='purple', linestyle=':', linewidth=1, alpha=0.7,
                 label='Residual (error))' if i == 0 else "_nolegend_")

        # Trazar línea tangente
        if i < len(tangents): # La lista de tangentes es una más corta que x_approxs
            info = tangents[i]
            x_tangent_points = np.linspace(x_min_plot, x_max_plot, 100)
            y_tangent_points = info['slope'] * (x_tangent_points - info['x_point']) + info['y_point']

            plt.plot(x_tangent_points, y_tangent_points, color='green', linestyle='--', linewidth=1,
                     label=f'Tangente en $x_{i}$' if i == 0 else "_nolegend_", alpha=0.7)

            # Dibujar el punto donde la tangente intersecta el eje x (siguiente aproximación)
            if i < len(x_approxs) - 1: # Si hay una siguiente aproximación
                next_x_val = x_approxs[i+1]
                plt.scatter(next_x_val, 0, color='orange', s=60, zorder=5, marker='x',
                            label=f'$x_{i+1}$ (Siguiente aprox.)' if i==0 else "_nolegend_", edgecolors='black')
                plt.text(next_x_val, -0.4, f'$x_{i+1}$', fontsize=10, ha='center', va='top', color='orange')

    # Añadir evaluación de la última aproximación de f(x)
    if x_approxs:
        final_x = x_approxs[-1]
        final_f_x = f(final_x)
        plt.text(x_max_plot - 0.5, y_plot_max - 0.5,
                 f'f($x_{len(x_approxs)-1}$): {final_f_x:.6f}',
                 fontsize=12, color='darkblue', ha='right', va='top',
                 bbox=dict(boxstyle="round,pad=0.3", fc="yellow", ec="b", lw=1, alpha=0.7))


    plt.title(f"Método de Newton Interactivo (Iteraciones: {iterations}, Iteración Inicial: {initial_guess})")
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.grid(True)

    # Leyenda personalizada para evitar etiquetas duplicadas del bucle
    handles, labels = plt.gca().get_legend_handles_labels()
    unique_labels = dict(zip(labels, handles)) # Usar diccionario para mantener solo etiquetas únicas
    plt.legend(unique_labels.values(), unique_labels.keys(), loc='best')

    plt.ylim(y_plot_min, y_plot_max)
    plt.xlim(x_min_plot, x_max_plot)
    plt.show()

# @title
# Crear deslizadores interactivos
interact(plot_newton,
         iterations=IntSlider(min=1, max=15, step=1, value=1, description='Iteraciones (N)', continuous_update=False),
         initial_guess=FloatSlider(min=-5.0, max=5.0, step=0.1, value=5.0, description='Iteración Inicial $x_0$', continuous_update=False)
        );